## Imports and constants

In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from lat_lon_parser import parse

# database connection with credentials
load_dotenv()
connection_string = os.getenv("CON_STRING")

# for scraping Wikipedia
headers = {'User-Agent': 'Chrome/134.0.0.0'}

## Goals
Now that you've learnt how to send and retrieve information, it's your turn to show off your skills. Create multiple tables in SQL for the data you scrapped about cities from Wikipedia. One should just be a table about the cities, the others should be facts about the cities.

| city_id | name | country | latitude | longitude |
| --- | --- | --- | --- | --- |
| 1 | Berlin | Germany | 52.52 | 13.41 |
| 2 | Hamburg | Germany | 53.55 | 10.00 |
| 3 | Munich | Germany | 48.14 | 11.58 |

<br>

| city_id | population | date_gathered |
|---|---|---|
| 1 | 3596999 | 2026-10-15 |
| 2 | 1973896 | 2026-10-15 |
| 3 | 1505036 | 2026-10-15 |

> **Pro Tip:** Visualise your relational database with pen and paper before you start coding. This can help you to identify any potential problems or inconsistencies in your design, and it can also make the coding process more efficient.

## Schema definition

See `/sql_files/gans_schema.sql`

## Data to write

Before writing to the SQL database, we'll need to repeat yesterday's web-scraping code to gather the data again.

### Functions

In [2]:
def city_info(cities):
    info_dict = {
        "name": [],
        "country": [],
        "latitude": [],
        "longitude": []
    }
    
    for city in cities:
        url = f"https://en.wikipedia.org/wiki/{city}"
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            city_soup = BeautifulSoup(response.content, 'html.parser')
            country_raw = city_soup.find("th", string="Country").find_next('td') 
            country = country_raw.get_text().strip().split("[")[0] 
            info_dict["country"].append(country)
        
            lat = parse(city_soup.find(class_="latitude").get_text())
            lon = parse(city_soup.find(class_="longitude").get_text())
            info_dict["latitude"].append(lat)
            info_dict["longitude"].append(lon)

            info_dict["name"].append(city)
        else:
            print(f"Failed to retrieve page for {city}: {response.status_code}")

    return pd.DataFrame(info_dict)

In [9]:
def pop_info(cities_df):
    populations = []
    city_ids = []
    for _, city in cities_df.iterrows():
        city_name = city["name"]
        url = f"https://en.wikipedia.org/wiki/{city_name}"
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            city_soup = BeautifulSoup(response.content, 'html.parser')
            for row in city_soup.find("table").find_all("tr"):
                if row.find(string="Population"):
                    population = int(row.find_next("td").get_text().replace(",", ""))
        populations.append(population)
        city_ids.append(city["city_id"])

    return pd.DataFrame({"city_id": city_ids, "population": populations, "date_gathered": pd.Timestamp.now().date()})

### Data gathering and recording

Because `populations` table references the auto-generated `city_id` in `cities`, we'll first write `cities_df` to the database and read it back before creating `pops_df`.

In [4]:
cities = ["Berlin", "Hamburg", "Munich"]

In [5]:
cities_df = city_info(cities)

cities_df.to_sql("cities",
                 if_exists="append",
                 con=connection_string,
                 index=False
                )

3

In [ ]:
cities_df = pd.read_sql("cities", con=connection_string)
pops_df = pop_info(cities_df)

pops_df.to_sql("populations",
               if_exists="append",
               con=connection_string,
               index=False
              )